# Sportmonks Datenakquise — Matchstatistiken Super League

Ergänzt die API-Football Basisdaten um **detaillierte Matchstatistiken** (Sportmonks Football API v3).

**Zusätzliche Daten pro Spiel & Team:**  
🔵 Ballbesitz · 🎯 Schüsse · 🟨 Karten · 🚩 Eckbälle · 💢 Fouls · ↗️ Abseits · 🧤 Saves · ⚡ Angriffe · 🔁 Pässe

**Voraussetzung** — `.env` im Projekt-Root:
```
API_SPORTMONKS_KEY="dein_key"
API_SPORTMONKS_URL="https://api.sportmonks.com/v3/football"
```

**Output-Dateien in `data_acquisition/raw/`:**
| Datei | Inhalt | Verwendung |
|---|---|---|
| `teams_sportmonks.csv` | Team-IDs & Metadaten | Referenz |
| `team_season_stats.csv` | Aggregierte Saison-Totals pro Team | Radar Chart, Scatter Plot |
| `fixture_statistics.csv` | Spiel-für-Spiel Stats (eine Zeile pro Team pro Spiel) | Heatmap |
| `season_timeline.csv` | Kumulierte Punkte pro Spieltag | Liniendiagramm |
| `teams_combined.csv` | Sportmonks + API-Football gejoined | Scatter: Ballbesitz vs. Punkte |

---
**v3-Besonderheiten:**
- Scores: `description` (`CURRENT`/`1ST_HALF`/`2ND_HALF`) + `score.participant` (`home`/`away`)
- State: `include=state` → Objekt mit `short_name` (`FT`, `NS`, `AET`, …)
- Statistiken: `type_id`-basiert, nur erfasste Werte, via `location: home|away`
- Season ID 25607 = Swiss Super League 2024/2025 (via Postman-Collection bestätigt)

## 1. Setup & Imports

In [ ]:
import os
import time
import requests
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv

env_path = Path("__file__").resolve().parent.parent / ".env"
load_dotenv(dotenv_path=env_path)

API_KEY  = os.environ["API_SPORTMONKS_KEY"]
BASE_URL = os.environ.get("API_SPORTMONKS_URL", "https://api.sportmonks.com/v3/football").rstrip("/")

if "docs.sportmonks" in BASE_URL:
    raise ValueError(
        "API_SPORTMONKS_URL zeigt auf die Doku, nicht die API!\n"
        "Korrekt: API_SPORTMONKS_URL=https://api.sportmonks.com/v3/football"
    )

HEADERS = {"Authorization": API_KEY}
RAW_DIR = Path("raw")
RAW_DIR.mkdir(exist_ok=True)

print(f"Base URL : {BASE_URL}")
print(f"API Key  : {'✓ geladen' if API_KEY else '✗ FEHLT'}")
print(f"Output   : {RAW_DIR.resolve()}")

## 2. Hilfsfunktionen

In [ ]:
def api_get(endpoint: str, params: dict = None) -> dict:
    """Einzelner GET-Request an Sportmonks v3."""
    url  = f"{BASE_URL}/{endpoint.lstrip('/')}"
    resp = requests.get(url, headers=HEADERS, params=params or {})
    resp.raise_for_status()
    return resp.json()


def api_get_all(endpoint: str, params: dict = None) -> list:
    """Paginierter GET: lädt alle Seiten (via has_more) und gibt flache Liste zurück."""
    params = {**(params or {}), "per_page": 50}
    all_data, page = [], 1
    while True:
        data      = api_get(endpoint, {**params, "page": page})
        items     = data.get("data", [])
        all_data.extend(items)
        rate_rem  = data.get("rate_limit", {}).get("remaining", "?")
        has_more  = data.get("pagination", {}).get("has_more", False)
        print(f"  Seite {page:>2} | +{len(items):>3} Einträge | Rate verbleibend: {rate_rem}")
        if not has_more:
            break
        page += 1
        time.sleep(0.4)
    return all_data


def parse_participants(participants: list) -> tuple[dict, dict]:
    """Gibt (home_team, away_team) zurück. Location steht in participant.meta.location."""
    home = {"team_id": None, "team_name": ""}
    away = {"team_id": None, "team_name": ""}
    for p in participants or []:
        loc   = (p.get("meta") or {}).get("location", "")
        entry = {"team_id": p["id"], "team_name": p.get("name", "")}
        if loc == "home":  home = entry
        elif loc == "away": away = entry
    return home, away


def parse_scores(scores: list) -> dict:
    """Extrahiert Tore für alle Abschnitte aus dem scores-Array.

    v3: Zwei Einträge pro description (1ST_HALF / 2ND_HALF / CURRENT),
    jeweils einer mit score.participant='home' und einer mit 'away'.
    CURRENT = Endstand bei abgeschlossenen Spielen.
    """
    result = {}
    for sc in scores or []:
        desc        = sc.get("description", "").upper()
        score_obj   = sc.get("score", {})
        participant = score_obj.get("participant", "")
        goals       = score_obj.get("goals")
        if desc in {"CURRENT", "1ST_HALF", "2ND_HALF"} and participant in ("home", "away"):
            result[f"score_{desc.lower()}_{participant}"] = goals
    result["goals_home"] = result.get("score_current_home")
    result["goals_away"] = result.get("score_current_away")
    return result


def parse_state(state_field) -> str:
    """Gibt short_name zurück (FT / NS / AET …). Fallback auf developer_name."""
    if isinstance(state_field, dict):
        return state_field.get("short_name", "") or state_field.get("developer_name", "")
    return ""


def parse_fixture_statistics(statistics: list, type_map: dict) -> tuple[dict, dict]:
    """Pivotiert type_id-basierte Fixture-Stats in (home_stats, away_stats).
    v3: Nur erfasste Werte; Felder via location: home|away.
    """
    home_stats, away_stats = {}, {}
    for stat in statistics or []:
        type_id  = stat.get("type_id")
        location = stat.get("location", "")
        value    = (stat.get("data") or {}).get("value")
        col_name = type_map.get(type_id, f"stat_type_{type_id}")
        if location == "home":  home_stats[col_name] = value
        elif location == "away": away_stats[col_name] = value
    return home_stats, away_stats


print("✓ Hilfsfunktionen bereit")

## 3. Statistik-Typ-Mapping (type_id → Spaltenname)

Bekannte IDs hardcoded. Unbekannte werden nach dem Fixtures-Laden via `GET /types/{id}` aufgelöst.

In [ ]:
def make_col_name(name: str) -> str:
    return (
        name.lower()
        .replace(" ", "_").replace("/", "_").replace("-", "_")
        .replace("(", "").replace(")", "").replace("%", "pct")
        .replace(".", "").replace("'", "")
    )


# Offizielle Sportmonks v3 Statistik-Typ-IDs.
# Quelle: https://docs.sportmonks.com/v3/definitions/types/statistics/team-statistics
# Lookups für unbekannte IDs: Core API GET /v3/core/types/{id} (NICHT /v3/football/types → 404)
TYPE_MAP: dict[int, str] = {
    # --- Offizielle Team-Stats-Typen (verifiziert via Sportmonks-Doku) ---
    34:    "corners",
    43:    "attacks",
    44:    "dangerous_attacks",
    45:    "ball_possession_pct",
    47:    "penalties",
    51:    "offsides",
    52:    "goals",
    56:    "fouls",
    78:    "tackles",
    83:    "red_cards",
    84:    "yellow_cards",
    85:    "yellowred_cards",
    88:    "goals_conceded",
    118:   "rating",
    194:   "cleansheets",
    214:   "wins",
    215:   "draws",
    216:   "losses",
    1677:  "shots",
    5304:  "expected_goals",
    27260: "injury_time_goals",
    27263: "games_played",
    # --- Fixture-Stats-Typen (aus Fixture-Endpoint beobachtet) ---
    41:  "shots_total",
    42:  "shots_on_target",
    53:  "passes_pct",
    57:  "offsides_count",
    58:  "saves",
    80:  "passes_total",
    81:  "passes_accurate",
}


def discover_unknown_types(type_ids: set, existing_map: dict) -> dict:
    """Löst unbekannte type_ids via Sportmonks Core API auf.

    WICHTIG: Types-Endpoint ist Teil der Core API (/v3/core/types/{id}),
    NICHT der Football API (/v3/football/types/{id} → gibt 404).
    """
    unknown  = sorted(tid for tid in type_ids if tid not in existing_map and tid is not None)
    enriched = dict(existing_map)
    if not unknown:
        print(f"  Alle {len(type_ids)} Type-IDs bereits bekannt ✓")
        return enriched
    core_url = "https://api.sportmonks.com/v3/core"
    print(f"  {len(unknown)} unbekannte Type-IDs → Core API GET /v3/core/types/{{id}}...")
    for tid in unknown:
        try:
            resp = requests.get(f"{core_url}/types/{tid}", headers=HEADERS)
            resp.raise_for_status()
            obj  = resp.json().get("data") or {}
            name = obj.get("name") or obj.get("developer_name") or f"stat_type_{tid}"
            enriched[tid] = make_col_name(name)
            print(f"    {tid:>6}: {enriched[tid]}")
        except Exception:
            enriched[tid] = f"stat_type_{tid}"
            print(f"    {tid:>6}: ❌ → stat_type_{tid}")
        time.sleep(0.2)
    return enriched


print(f"TYPE_MAP bereit: {len(TYPE_MAP)} bekannte Typen vorgeladen.")

## 4. Season ID & Teams — Swiss Super League 2024/2025

In [ ]:
# Season ID 25607 = Swiss Super League 2024/2025
# Bestätigt via Postman-Collection (https://postman.sportmonks.com)
SEASON_ID = 25607

print(f"Saison: Swiss Super League 2024/2025  (ID {SEASON_ID})\n")
try:
    s = api_get(f"seasons/{SEASON_ID}").get("data", {})
    print(f"✅ {s.get('name', '?')}  ({s.get('starting_at', '?')} → {s.get('ending_at', '?')})")
except Exception as e:
    print(f"⚠️  Saison-Check fehlgeschlagen: {e}")

# Alle Teams in der Saison → Team-IDs für Statistik-Requests
print(f"\nLade alle Teams der Saison {SEASON_ID}...")
teams_raw = api_get_all(f"teams/seasons/{SEASON_ID}")

df_teams_sm = pd.DataFrame([
    {
        "team_id":   t["id"],
        "team_name": t.get("name", ""),
        "short_name": t.get("short_name", ""),
        "team_code": t.get("code", ""),
        "founded":   t.get("founded"),
        "logo":      t.get("image_path", ""),
    }
    for t in teams_raw
])
df_teams_sm.to_csv(RAW_DIR / "teams_sportmonks.csv", index=False)
TEAM_IDS = df_teams_sm["team_id"].tolist()

print(f"\n✅ {len(df_teams_sm)} Teams gespeichert:")
print(df_teams_sm[["team_id", "team_name", "short_name"]].to_string(index=False))

## 5. Saison-Statistiken pro Team (direkte API-Totals)

**Endpoint:** `GET /statistics/seasons/teams/{team_id}?filters=seasonId:25607&include=details`  
Liefert aggregierte Saison-Totals direkt — kein manuelles Aggregieren nötig.  
Response: `details[]` mit `type_id` + `value`-Objekt (Struktur variiert je Stat-Typ).

In [ ]:
def parse_stat_value(value) -> float | None:
    """Extrahiert den relevantesten Wert aus dem value-Objekt.

    Mögliche Strukturen:
      {"total": 45}                    → total
      {"average": "2.50", "total": 25} → total bevorzugt
      {"goals": 3, "penalties": 1}     → goals
      {"highest": 90, "lowest": 10}    → highest
    """
    if value is None:
        return None
    if isinstance(value, (int, float)):
        return float(value)
    if isinstance(value, dict):
        for key in ("total", "goals", "average", "highest", "value"):
            if key in value and value[key] is not None:
                try:
                    return float(value[key])
                except (TypeError, ValueError):
                    pass
    return None


def fetch_team_season_stats(team_id: int, season_id: int, type_map: dict) -> dict:
    """Holt aggregierte Saison-Stats für ein Team.

    Der Endpoint gibt ALLE Saisons eines Teams zurück.
    Filter 'IdAfter:{season_id-1}' begrenzt auf aktuelle Saison.
    Anschliessend clientseitiger Match auf season_id zur Sicherheit.

    Falscher Filter (400-Fehler): filters=seasonId:25607
    Korrekter Filter:             filters=IdAfter:25606
    """
    try:
        resp = api_get(
            f"statistics/seasons/teams/{team_id}",
            params={
                "include": "details",
                "filters": f"IdAfter:{season_id - 1}",
            }
        )
    except Exception as e:
        print(f"    ⚠️  {e}")
        return {}

    data    = resp.get("data", [])
    records = data if isinstance(data, list) else [data]

    record = next(
        (r for r in records if r.get("season_id") == season_id),
        records[0] if records else None
    )
    if not record:
        print(f"    ⚠️  Kein Eintrag für season_id={season_id}")
        return {}

    return {
        type_map.get(d.get("type_id"), f"stat_type_{d.get('type_id')}"): parse_stat_value(d.get("value"))
        for d in record.get("details", [])
        if parse_stat_value(d.get("value")) is not None
    }


assert TEAM_IDS, "TEAM_IDS nicht gesetzt — Zelle 4 zuerst ausführen!"
print(f"Lade Saison-Statistiken für {len(TEAM_IDS)} Teams (Season {SEASON_ID})...\n")
print(f"Endpoint: GET /statistics/seasons/teams/{{id}}?filters=IdAfter:{SEASON_ID - 1}&include=details\n")

team_season_rows = []
for team_id in TEAM_IDS:
    name  = df_teams_sm.loc[df_teams_sm["team_id"] == team_id, "team_name"].values[0]
    print(f"  → {name} (ID {team_id})")
    stats = fetch_team_season_stats(team_id, SEASON_ID, TYPE_MAP)
    team_season_rows.append({"team_id": team_id, "team_name": name, **stats})
    time.sleep(0.4)

df_team_season = pd.DataFrame(team_season_rows)

# --- Unbekannte type_ids sofort auflösen, bevor wir speichern ---
unknown_cols = [c for c in df_team_season.columns if c.startswith("stat_type_")]
if unknown_cols:
    unknown_ids = {int(c.replace("stat_type_", "")) for c in unknown_cols}
    print(f"\n  {len(unknown_ids)} unbekannte type_ids: {sorted(unknown_ids)}")
    print("  Löse via GET /types/{id} auf...")
    TYPE_MAP = discover_unknown_types(unknown_ids, TYPE_MAP)
    rename_map = {
        col: TYPE_MAP.get(int(col.replace("stat_type_", "")), col)
        for col in unknown_cols
        if TYPE_MAP.get(int(col.replace("stat_type_", "")), col) != col
    }
    if rename_map:
        df_team_season = df_team_season.rename(columns=rename_map)
        print(f"  ✓ Spalten umbenannt: {rename_map}")
    else:
        print("  ⚠️  Keine Namen via API gefunden — Spalten bleiben stat_type_*")
else:
    print("\n  Alle type_ids bereits bekannt ✓")

df_team_season.to_csv(RAW_DIR / "team_season_stats.csv", index=False)

stat_cols = [c for c in df_team_season.columns if c not in {"team_id", "team_name"}]
print(f"\n✅ team_season_stats.csv gespeichert")
print(f"   {len(df_team_season)} Teams × {len(df_team_season.columns)} Spalten")
print(f"   Spalten: {', '.join(stat_cols)}")
df_team_season

## 6. Visualisierungen — FC Thun im Ligavergleich

Vier Übersichtsplots auf Basis der `team_season_stats.csv`.  
FC Thun ist jeweils **rot** hervorgehoben, der Ligadurchschnitt als gestrichelte Linie.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

# --- Farben & Daten ---
THUN_COLOR = "#E30613"   # FC Thun Rot
BASE_COLOR = "#BBBBBB"
MEAN_COLOR = "#1A3A6B"   # Dunkelblau für Ø-Linie

FIG_DIR = Path("../eda/reports")
FIG_DIR.mkdir(parents=True, exist_ok=True)

df_viz = pd.read_csv(RAW_DIR / "team_season_stats.csv")
df_viz["is_thun"] = df_viz["team_name"].str.contains("Thun", case=False, na=False)

def bar_colors(df, sort_col, ascending=True):
    d = df.sort_values(sort_col, ascending=ascending).reset_index(drop=True)
    return d, [THUN_COLOR if t else BASE_COLOR for t in d["is_thun"]]

# ─── Figur: 2×2 ────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(16, 13))
fig.suptitle("FC Thun — Super League 2024/25 Saisonübersicht", fontsize=17, fontweight="bold", y=1.01)

# ── Plot 1: Ballbesitz % ────────────────────────────────────────────────────
ax1 = fig.add_subplot(2, 2, 1)
d1, c1 = bar_colors(df_viz, "ball_possession_pct")
bars = ax1.barh(d1["team_name"], d1["ball_possession_pct"], color=c1, edgecolor="white", height=0.7)
avg1 = d1["ball_possession_pct"].mean()
ax1.axvline(avg1, color=MEAN_COLOR, linestyle="--", linewidth=1.5, label=f"Ø Liga {avg1:.1f}%")
ax1.set_title("Ballbesitz (%)", fontweight="bold")
ax1.set_xlabel("Ballbesitz %")
ax1.legend(fontsize=9)
ax1.xaxis.set_major_formatter(mticker.FormatStrFormatter("%.0f%%"))

# ── Plot 2: Schüsse Total ───────────────────────────────────────────────────
ax2 = fig.add_subplot(2, 2, 2)
d2, c2 = bar_colors(df_viz, "shots")
ax2.barh(d2["team_name"], d2["shots"], color=c2, edgecolor="white", height=0.7)
avg2 = d2["shots"].mean()
ax2.axvline(avg2, color=MEAN_COLOR, linestyle="--", linewidth=1.5, label=f"Ø Liga {avg2:.0f}")
ax2.set_title("Schüsse Total (Saison)", fontweight="bold")
ax2.set_xlabel("Schüsse")
ax2.legend(fontsize=9)

# ── Plot 3: Scatter Ballbesitz vs. Schüsse ──────────────────────────────────
ax3 = fig.add_subplot(2, 2, 3)
for _, row in df_viz.iterrows():
    is_thun = row["is_thun"]
    ax3.scatter(
        row["ball_possession_pct"], row["shots"],
        color=THUN_COLOR if is_thun else BASE_COLOR,
        s=130 if is_thun else 60,
        zorder=5 if is_thun else 2,
        edgecolors="white", linewidths=0.5,
    )
    if is_thun or abs(row["shots"] - df_viz["shots"].mean()) > 40:
        ax3.annotate(
            row["team_name"],
            (row["ball_possession_pct"], row["shots"]),
            textcoords="offset points", xytext=(6, 4), fontsize=8,
            color=THUN_COLOR if is_thun else "#555555", fontweight="bold" if is_thun else "normal",
        )
ax3.axvline(df_viz["ball_possession_pct"].mean(), color=MEAN_COLOR, linestyle="--", linewidth=1, alpha=0.6)
ax3.axhline(df_viz["shots"].mean(),              color=MEAN_COLOR, linestyle="--", linewidth=1, alpha=0.6)
ax3.set_title("Ballbesitz vs. Schüsse", fontweight="bold")
ax3.set_xlabel("Ballbesitz %")
ax3.set_ylabel("Schüsse Total")

# ── Plot 4: Radar Thun vs. Ligadurchschnitt ─────────────────────────────────
radar_cols   = ["ball_possession_pct", "shots", "dangerous_attacks", "tackles", "corners", "fouls"]
radar_labels = ["Ballbesitz %", "Schüsse", "Gef. Angriffe", "Tackles", "Eckbälle", "Fouls"]

thun_row   = df_viz[df_viz["is_thun"]].iloc[0]
league_avg = df_viz[radar_cols].mean()
mins       = df_viz[radar_cols].min()
maxs       = df_viz[radar_cols].max()
thun_norm  = ((thun_row[radar_cols] - mins) / (maxs - mins + 1e-9)).values
avg_norm   = ((league_avg           - mins) / (maxs - mins + 1e-9)).values

N      = len(radar_cols)
angles = [n / N * 2 * np.pi for n in range(N)] + [0]
t_vals = list(thun_norm) + [thun_norm[0]]
a_vals = list(avg_norm)  + [avg_norm[0]]

ax4 = fig.add_subplot(2, 2, 4, polar=True)
ax4.plot(angles, t_vals, color=THUN_COLOR, linewidth=2.5, label="FC Thun")
ax4.fill(angles, t_vals, color=THUN_COLOR, alpha=0.20)
ax4.plot(angles, a_vals, color=MEAN_COLOR, linewidth=1.5, linestyle="--", label="Ø Liga")
ax4.fill(angles, a_vals, color=MEAN_COLOR, alpha=0.08)
ax4.set_xticks(angles[:-1])
ax4.set_xticklabels(radar_labels, fontsize=9)
ax4.set_ylim(0, 1)
ax4.set_yticks([0.25, 0.5, 0.75, 1.0])
ax4.set_yticklabels(["25%", "50%", "75%", "100%"], fontsize=7, color="#888888")
ax4.set_title("Stärken-Profil\n(normalisiert, Ligavergleich)", fontweight="bold", pad=18)
ax4.legend(loc="upper right", bbox_to_anchor=(1.35, 1.15), fontsize=9)

plt.tight_layout()
out_path = FIG_DIR / "thun_season_overview.png"
plt.savefig(out_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"✅ Visualisierung gespeichert: {out_path}")

## 10. Übersicht aller gespeicherten Dateien

In [ ]:
print("Gespeicherte Dateien in data_acquisition/raw/:\n")
for f in sorted(RAW_DIR.glob("*.csv")):
    df   = pd.read_csv(f)
    size = f.stat().st_size / 1024
    print(f"  {f.name:<38} {len(df):>4} Zeilen × {len(df.columns):>3} Spalten  ({size:.1f} KB)")

print("\nVisualisierungen in eda/reports/:")
for f in sorted(Path("../eda/reports").glob("*.png")):
    size = f.stat().st_size / 1024
    print(f"  {f.name:<38} ({size:.1f} KB)")

print("\n🏁 Sportmonks Datenakquise abgeschlossen.")
print("   Nächster Schritt: uv run python eda/generate-data-profile.py")